# Part B — PySpark Loading, Cleaning & Partitioning

**Run this AFTER `01_Parsing_Timetables_Fares_Disruptions.ipynb`** -- it depends on `timetables_flat.csv` and `fares_flat.csv` which that notebook creates.


## Part B — PySpark Loading, Cleaning & Partitioning

Now we move the three flattened tables into PySpark for the large-scale cleaning, partitioning, caching, and (later) joins/ML — satisfying the brief's PySpark requirement.

### 6. Start Spark session

In [ ]:
import os
os.environ["PYSPARK_PYTHON"] = "python"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

spark = SparkSession.builder \
    .appName("FareReliabilityAnalysis") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark

### 7. Load the three flattened files into Spark DataFrames

In [ ]:
timetables_schema = StructType([
    StructField("service_code", StringType(), True),
    StructField("line_name", StringType(), True),
    StructField("operator_noc", StringType(), True),
    StructField("vehicle_journey_code", StringType(), True),
    StructField("direction", StringType(), True),
    StructField("days_of_week", StringType(), True),
    StructField("stop_sequence", IntegerType(), True),
    StructField("stop_point_ref", StringType(), True),
    StructField("stop_name", StringType(), True),
    StructField("scheduled_time_sec", DoubleType(), True),
    StructField("source_file", StringType(), True),
    StructField("scheduled_time", StringType(), True),
])

timetables_spark = spark.read.csv("timetables_flat.csv", header=True, schema=timetables_schema)

fares_schema = StructType([
    StructField("operator", StringType(), True),
    StructField("route", StringType(), True),
    StructField("direction", StringType(), True),
    StructField("ticket_type", StringType(), True),
    StructField("price_gbp", DoubleType(), True),
    StructField("source_file", StringType(), True),
])

fares_spark = spark.read.csv("fares_flat.csv", header=True, schema=fares_schema)

disruptions_spark = spark.read.csv("catalogue/disruptions_data_catalogue.csv", header=True, inferSchema=True)
disruptions_spark = disruptions_spark.filter(F.col("Organisation") == "West of England")

print("Timetables rows:", timetables_spark.count())
print("Fares rows:", fares_spark.count())
print("Disruptions rows:", disruptions_spark.count())

### 8. Clean each DataFrame

In [ ]:
# --- Timetables cleaning ---
timetables_clean = (
    timetables_spark
    .filter(F.col("stop_point_ref").isNotNull())
    .filter(F.col("line_name").isNotNull())
    .withColumn("line_name", F.trim(F.col("line_name")))
    .dropDuplicates(["service_code", "vehicle_journey_code", "stop_sequence"])
)
print("Timetables - raw:", timetables_spark.count(), "| clean:", timetables_clean.count())

# --- Fares cleaning ---
fares_clean = (
    fares_spark
    .filter(F.col("price_gbp").isNotNull())
    .filter(F.col("price_gbp") > 0)
    .withColumn("route", F.trim(F.col("route")))
    .dropDuplicates(["operator", "route", "direction", "ticket_type"])
)
print("Fares - raw:", fares_spark.count(), "| clean:", fares_clean.count())

# --- Disruptions cleaning ---
disruptions_clean = (
    disruptions_spark
    .filter(F.col("Services affected").isNotNull())
    .withColumnRenamed("Services affected", "route")
    .withColumn("route", F.trim(F.col("route").cast("string")))
)
print("Disruptions (West of England) - raw:", disruptions_spark.count(), "| clean:", disruptions_clean.count())

### 9. Partitioning + Caching (Big Data requirement)

Repartition Timetables into at least 4 partitions and cache — required for the brief's Spark UI evidence.

In [ ]:
timetables_clean = timetables_clean.repartition(8, "line_name")
timetables_clean.cache()
fares_clean.cache()
disruptions_clean.cache()

print("Timetables partitions:", timetables_clean.rdd.getNumPartitions())

In [ ]:
timetables_clean.show(5, truncate=False)
fares_clean.show(5, truncate=False)
disruptions_clean.show(5, truncate=False)

### 10. Save cleaned DataFrames as Parquet (efficient format for later joins/ML)

In [ ]:
timetables_clean.write.mode("overwrite").parquet("output/timetables_clean.parquet")
fares_clean.write.mode("overwrite").parquet("output/fares_clean.parquet")
disruptions_clean.write.mode("overwrite").parquet("output/disruptions_clean.parquet")

print("Cleaning complete. Parquet files written to /output.")

## Next steps

- Join Timetables <-> Fares on `route`
- Join Timetables <-> Disruptions on `route`
- Engineer a reliability feature (e.g. disruption count per route, or trips-per-route vs disruptions-per-route ratio)
- Confirm total merged row count against the 100,000-record threshold (already well exceeded from Timetables alone: 500k+ rows)
- Move to feature engineering + ML modelling (regression: fare -> reliability score)
